In [4]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/data")

print("Data folder exists:", DATA_DIR.exists())
print("Files/folders:")

for item in DATA_DIR.iterdir():
    print(item)

Data folder exists: True
Files/folders:
/content/drive/MyDrive/data/raw
/content/drive/MyDrive/data/processed
/content/drive/MyDrive/data/embeddings


In [6]:
PROCESSED_DIR = DATA_DIR / "processed"

FULL_FILE = PROCESSED_DIR / "documents.jsonl"
DEV_FILE = PROCESSED_DIR / "documents_dev.jsonl"

print("Full dataset exists:", FULL_FILE.exists())
print("Dev dataset exists:", DEV_FILE.exists())

Full dataset exists: True
Dev dataset exists: True


In [7]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

In [8]:
documents = []

with open(FULL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

ids = [doc["id"] for doc in documents]
texts = [doc["text"] for doc in documents]

print("Total documents:", len(documents))
print("Total texts:", len(texts))
print("First document:")
print(documents[0])

Total documents: 119921
Total texts: 119921
First document:
{'id': 1, 'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.", 'metadata': {'category': 2}}


In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded")
print("Embedding dimension:", model.get_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded
Embedding dimension: 384


In [10]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

GPU available: True
GPU: Tesla T4


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Using device:", device)

Using device: cuda


In [12]:
embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

Batches:   0%|          | 0/937 [00:00<?, ?it/s]

Embedding shape: (119921, 384)
Data type: float32


In [13]:
from pathlib import Path
import json
import numpy as np

EMBEDDINGS_DIR = DATA_DIR / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

np.save(EMBEDDINGS_DIR / "embeddings.npy", embeddings)
np.save(EMBEDDINGS_DIR / "ids.npy", np.array(ids))

metadata = {
    "model": "all-MiniLM-L6-v2",
    "dimension": 384,
    "normalized": True,
    "document_count": len(documents)
}

with open(EMBEDDINGS_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Saved successfully!")
print("Location:", EMBEDDINGS_DIR)

Saved successfully!
Location: /content/drive/MyDrive/data/embeddings


In [14]:
print("Embeddings:", (EMBEDDINGS_DIR / "embeddings.npy").exists())
print("IDs:", (EMBEDDINGS_DIR / "ids.npy").exists())
print("Metadata:", (EMBEDDINGS_DIR / "metadata.json").exists())

Embeddings: True
IDs: True
Metadata: True
